# Plot Submission

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src" / "gridsearch_lgbm.py").exists():
            return p
    raise FileNotFoundError(
        "Could not locate repo root (src/gridsearch_lgbm.py not found above "
        f"{start}). Run this notebook from inside the MIGHTE-respicast-jointGBM checkout."
    )

ROOT_DIR = find_repo_root(Path.cwd())
print("Repo root:", ROOT_DIR)


In [ ]:
MODEL_ID = "ISI-LightGBM"
SUBMISSION_DIR = ROOT_DIR / "forecasts" / "prospective" / "submission"
DATA_FILE = ROOT_DIR / "data" / "processed" / "respicast_long_latest.csv"

ORIGIN_DATE = None  # e.g. "2026-08-12" to pin a specific week — None picks the latest available
WEEKS_HISTORY = 16   # how many past weeks of "observed" to show per panel

SAVE_FIGS = False
FIGS_DIR = ROOT_DIR / "forecasts" / "prospective" / "viz"


In [ ]:
candidates = sorted(SUBMISSION_DIR.glob(f"*-{MODEL_ID}.csv"))
if not candidates:
    raise FileNotFoundError(f"No '*-{MODEL_ID}.csv' submission found in {SUBMISSION_DIR}")

if ORIGIN_DATE is not None:
    matches = [p for p in candidates if p.name.startswith(str(ORIGIN_DATE))]
    if not matches:
        raise FileNotFoundError(
            f"No submission for origin_date={ORIGIN_DATE!r} in {SUBMISSION_DIR}. "
            f"Available: {[p.name for p in candidates]}"
        )
    submission_path = matches[0]
else:
    # Filenames are YYYY-MM-DD-<model-id>.csv, so lexicographic sort == chronological.
    submission_path = candidates[-1]

print(f"Using: {submission_path.name}")

combined = pd.read_csv(submission_path)
origin_date = str(combined["origin_date"].iloc[0])
targets_present = sorted(combined["target"].unique())
print(f"origin_date={origin_date}  targets={targets_present}  "
      f"locations={combined['location'].nunique()}  rows={len(combined)}")
combined.head()


In [ ]:
def plot_forecast_grid(combined, target_col, data_file, origin_date, weeks_history=16, ncols=5):
    truth = pd.read_csv(data_file)
    truth = truth[truth["target"] == target_col].copy()
    truth["truth_date"] = pd.to_datetime(truth["truth_date"])

    sub = combined[combined["target"] == target_col].copy()
    sub["target_end_date"] = pd.to_datetime(sub["target_end_date"])

    locations = sorted(sub["location"].unique())
    nrows = int(np.ceil(len(locations) / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 2.6 * nrows), sharex=False)
    axes = np.atleast_2d(axes)

    history_start = pd.to_datetime(origin_date) - pd.Timedelta(weeks=weeks_history)

    for i, loc in enumerate(locations):
        ax = axes[i // ncols, i % ncols]

        hist = truth[(truth["location"] == loc) & (truth["truth_date"] >= history_start)].sort_values("truth_date")
        ax.plot(hist["truth_date"], hist["value"], "-o", color="black", markersize=3, linewidth=1, label="observed")

        fc = sub[sub["location"] == loc]
        piv = fc.pivot(index="target_end_date", columns="output_type_id", values="value").sort_index()
        if not piv.empty:
            ax.plot(piv.index, piv[0.5], "-o", color="#E41A1C", markersize=3, linewidth=1, label="pred")
            if 0.025 in piv.columns and 0.975 in piv.columns:
                ax.fill_between(piv.index, piv[0.025], piv[0.975], color="#E41A1C", alpha=0.15)
            if 0.25 in piv.columns and 0.75 in piv.columns:
                ax.fill_between(piv.index, piv[0.25], piv[0.75], color="#E41A1C", alpha=0.25)

        ax.set_title(loc, fontsize=10)
        ax.tick_params(axis="x", rotation=45, labelsize=7)
        ax.tick_params(axis="y", labelsize=7)
        ax.grid(alpha=0.3)

    for j in range(len(locations), nrows * ncols):
        axes[j // ncols, j % ncols].axis("off")

    handles = [
        plt.Line2D([], [], color="black", marker="o", markersize=4, label="observed"),
        plt.Line2D([], [], color="#E41A1C", marker="o", markersize=4, label="pred"),
    ]
    fig.legend(handles=handles, loc="upper right", fontsize=9, title="Series")

    target_label = target_col.replace(" incidence", "").upper()
    fig.suptitle(f"{target_label} Incidence Predictions", fontsize=14, y=1.02)
    fig.tight_layout()
    return fig


In [ ]:
figs = {}
for target_col in targets_present:
    fig = plot_forecast_grid(combined, target_col, DATA_FILE, origin_date, weeks_history=WEEKS_HISTORY)
    figs[target_col] = fig
    if SAVE_FIGS:
        FIGS_DIR.mkdir(parents=True, exist_ok=True)
        slug = target_col.replace(" incidence", "").upper()
        out_path = FIGS_DIR / f"{origin_date}-{MODEL_ID}-{slug}-grid.png"
        fig.savefig(out_path, dpi=150, bbox_inches="tight")
        print(f"Saved: {out_path}")
    plt.show()
